# v0.2.1 · Notebook 04: Dataset card and publishing

**LLM Security Gateway** · writes the dataset card from the real data and publishes the dataset to Hugging Face.

*v0.2.1:* republishes the fixed dataset over the public one. The old version stays downloadable under the tag `v0.2`, and the new one is tagged `v0.2.1`.

1. Loads `v02_dataset.zip` from notebook 02 (you'll be asked to upload it)
2. Runs final sanity checks (labels, empty texts, leakage between splits)
3. Builds the dataset card: **every number in it is computed from the data**, so the card can't disagree with the dataset
4. Tags the current public version as `v0.2`, publishes the new data, tags it `v0.2.1`

**Runtime: CPU is enough.** Needs the `HF_TOKEN` secret you set up in Colab (🔑 icon, *Acceso del cuaderno* on).

In [ ]:
!pip -q install -U datasets huggingface_hub

In [ ]:
import os, re, unicodedata, zipfile
import pandas as pd

REPO_ID = "edithngalame/prompt-injection-en-es"
GITHUB = "https://github.com/edithngalame/llm-security-gateway"
PRIVATE = False         # v0.2.1: the dataset is already public
VERSION = "v0.2.1"
PREVIOUS = "v0.2"
OUT_DIR = "/content/v02"
pd.set_option("display.max_colwidth", 120)

## 1. Load the dataset from notebook 02

In [ ]:
PARQUET = f"{OUT_DIR}/dataset_v0.2.parquet"
if not os.path.exists(PARQUET):
    from google.colab import files
    print("Upload v02_dataset.zip from notebook 02:")
    up = files.upload()
    os.makedirs(OUT_DIR, exist_ok=True)
    zipfile.ZipFile(next(iter(up))).extractall(OUT_DIR)
df = pd.read_parquet(PARQUET)
df["split"] = df["split"].replace({"val": "validation"})
df = df[["text", "label", "source", "lang", "family", "origin", "license", "split"]]
print(f"{len(df):,} rows")
display(df["split"].value_counts())

## 2. Final checks before publishing

In [ ]:
def dedup_key(t):
    t = unicodedata.normalize("NFKC", t).lower()
    return re.sub(r"\s+", " ", re.sub(r"[^\w]+", " ", t)).strip()

checks = {
    "no empty texts": (df.text.str.strip().str.len() > 0).all(),
    "labels are 0/1": set(df.label.unique()) <= {0, 1},
    "every row has a split": df.split.notna().all(),
    "every row has a licence": df.license.notna().all(),
    "validation has harmless emails": ((df.split == "validation") & (df.label == 0)
                                       & df.text.str.contains("SUBJECT:", regex=False)).any(),
    "no text in two splits": (df.assign(k=df.text.map(dedup_key)).groupby("k").split.nunique() <= 1).all(),
}
for name, ok in checks.items():
    print(("PASS " if ok else "FAIL ") + name)
assert all(checks.values()), "fix the failing check before publishing"

## 3. Build the dataset card

In [ ]:
SPLIT_NAMES = {"val": "validation"}  # Hugging Face's conventional name

SPLIT_INFO = {
    "train": "Training data.",
    "validation": "For choosing thresholds and checkpoints. Never report final results on it.",
    "test": "Main test set, same distribution as training.",
    "test_heldout": "Emails with attack instructions from BIPIA's *test* list: attack styles never seen in training.",
    "test_native_es": "Human-translated and adapted Spanish test set, written by a Spanish speaker. Never used for training.",
    "test_other_lang": "German, French, Italian and Portuguese texts: a cross-language test.",
    "eval_handwritten": "Hand-written demo and red-team messages from the project repository.",
}

SOURCES = [
    ("deepset/prompt-injections", "https://huggingface.co/datasets/deepset/prompt-injections", "Apache-2.0", "direct attacks + harmless"),
    ("Lakera/gandalf_ignore_instructions", "https://huggingface.co/datasets/Lakera/gandalf_ignore_instructions", "MIT", "direct attacks from Gandalf players"),
    ("jackhhao/jailbreak-classification", "https://huggingface.co/datasets/jackhhao/jailbreak-classification", "Apache-2.0", "role-play jailbreaks + harmless"),
    ("microsoft/BIPIA (email part)", "https://github.com/microsoft/BIPIA", "MIT", "indirect attacks inserted into emails"),
    ("argilla/databricks-dolly-15k-curated-multilingual", "https://huggingface.co/datasets/argilla/databricks-dolly-15k-curated-multilingual", "CC-BY-SA-3.0", "harmless questions and documents, EN + ES"),
    ("Helsinki-NLP/opus-mt-en-es", "https://huggingface.co/Helsinki-NLP/opus-mt-en-es", "Apache-2.0", "machine translation to Spanish"),
]


def _table(rows, header):
    out = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    out += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    return "\n".join(out)


def split_table(df: pd.DataFrame) -> str:
    rows = []
    for name in SPLIT_INFO:
        g = df[df["split"] == name]
        if len(g):
            rows.append((f"`{name}`", f"{len(g):,}", f"{int((g.label == 1).sum()):,}",
                         f"{int((g.label == 0).sum()):,}", SPLIT_INFO[name]))
    return _table(rows, ["Split", "Rows", "Injection", "Benign", "Purpose"])


def shortcut_lines(df: pd.DataFrame) -> str:
    tr = df[df["split"] == "train"]
    es = tr[tr.lang == "es"]
    em = tr[tr.text.str.contains("SUBJECT:", regex=False)]
    return (f"- Spanish texts in training that are injections: **{(es.label == 1).mean():.0%}** "
            f"({len(es):,} rows). Before the fix: under 1%, which would teach \"Spanish = safe\".\n"
            f"- Email-shaped texts in training that are injections: **{(em.label == 1).mean():.0%}** "
            f"({len(em):,} rows). Before the fix: 96%, which would teach \"email = attack\". "
            f"Still above the overall rate, so false positives on harmless emails are measured separately.")


def build_card(df: pd.DataFrame, repo_id: str, github: str) -> str:
    n, n_att = len(df), int((df.label == 1).sum())
    langs = sorted(df.lang.dropna().unique())
    sources_md = _table([(f"[{a}]({u})", l, use) for a, u, l, use in SOURCES], ["Source", "Licence", "Used for"])
    return f"""---
license: other
license_name: mixed-per-row
license_link: https://huggingface.co/datasets/{repo_id}#sources-and-licences
language: [{", ".join(langs)}]
task_categories: [text-classification]
tags: [prompt-injection, llm-security, ai-security, jailbreak, multilingual, spanish]
pretty_name: Prompt Injection EN-ES (LLM Security Gateway v0.2.1)
size_categories: [10K<n<100K]
---

# Prompt Injection EN-ES

A multilingual (English + Spanish) dataset for detecting **prompt injection**: text that tries to make an AI system follow an attacker's instructions instead of its owner's. It covers **direct** attacks typed by users and **indirect** attacks hidden in documents such as emails.

Built for the [LLM Security Gateway]({github}) project. **{n:,} rows**, {n_att:,} injections and {n - n_att:,} benign.

## What makes it different

- **Source-aware**: every row says where the text entered the AI's context: typed by the `user` or inside a `retrieved` document.
- **Spanish**, including a small human-translated test set, not only machine translation.
- **Shortcut fixes, measured**: two patterns that would let a model cheat were found and corrected (see below).
- **Held-out evaluation sets** for attack styles and languages never seen in training.
- **Leakage-checked**: no text appears in more than one split, translations always share the split of their English original, and every version of an email (attacked, clean, harmless) stays in one split.

## Labels

- `1` **injection**: text trying to make the AI follow the author's instructions instead of its owner's (override rules, extract hidden instructions, hijack actions, manipulate answers).
- `0` **benign**: everything else, including *hard benign* text that talks *about* rules, instructions or AI without trying to hijack it.

## Fields

| Field | Description |
|---|---|
| `text` | The message or document |
| `label` | 1 = injection, 0 = benign |
| `source` | `user` (typed into the chat) or `retrieved` (inside a document the AI reads) |
| `lang` | Language code |
| `family` | Attack or text family (e.g. `gandalf`, `jailbreak_roleplay`, `bipia_*`, `hard_benign`) |
| `origin` | Dataset the row came from, marked `(MT-es: opus-mt-en-es)` when machine-translated |
| `license` | Licence of the original source for this row |

## Splits

{split_table(df)}

## Shortcut fixes

{shortcut_lines(df)}

## Sources and licences

{sources_md}

**Licensing is mixed and recorded per row** in the `license` column. Rows derived from Dolly carry CC-BY-SA-3.0 (share-alike); rows from the other sources carry their original permissive licences. Respect each row's licence when reusing it.

## How it was built

Reproducible Colab notebooks in the [project repository]({github}/tree/main/notebooks):

1. `01_collect_sources`: download, normalise, detect language, deduplicate, drop texts labelled both ways
2. `02_splits_and_spanish`: group-aware splits *before* augmentation, Spanish translation of attacks and matching benign texts, harmless email inserts, leakage check
3. `04_dataset_card_and_publish`: this card

## Known limitations

- **Most Spanish text is machine-translated.** The human-translated test set is small (32 rows) and translated from English rather than written from scratch.
- **Indirect attacks come from one source (BIPIA emails)** and reuse a limited set of attack instructions. Other document types (web pages, PDFs) are only covered by the hand-written sets.
- **No tool-output or MCP tool-description attacks yet.** The gateway handles tool risk through permissions and taint tracking instead (v0.4).
- **Language labels for some rows are automatic** (`langid`) and can be wrong for very short texts.
- **Harmless email sentences come from fixed lists**: 36 for training, 12 for validation and 12 for test, with no overlap. Real harmless emails are far more varied.
- **Residual email bias**: email-shaped texts are still more often injections than average.
- The benign side is mostly general questions and encyclopedic text; real application traffic will differ.

## Intended use

Training and evaluating prompt-injection detectors, especially multilingual and source-aware ones. **Not** intended as a complete security solution: detection should be combined with least-privilege design for AI agents.

## Changelog

- **v0.2.1**: `validation` now contains emails (20% of BIPIA's training emails, moved as whole emails). In v0.2 it had none, so thresholds tuned on it blocked about 9% of harmless test emails. Harmless email sentences are now split into separate train / validation / test lists (v0.2 reused the same 25 in every split, and a baseline memorised them).
- **v0.2**: first release. Still available: `load_dataset("{repo_id}", revision="v0.2")`.

## Citation

```bibtex
@misc{{ngalame2026promptinjectionenes,
  title  = {{Prompt Injection EN-ES: a source-aware multilingual prompt injection dataset}},
  author = {{Ngalame Atechkeng, Edith Dione}},
  year   = {{2026}},
  url    = {{https://huggingface.co/datasets/{repo_id}}}
}}
```
"""

In [ ]:
card_md = build_card(df, REPO_ID, GITHUB)
from IPython.display import Markdown
display(Markdown(card_md.split("---", 2)[2]))   # preview without the metadata header

## 4. Publish to Hugging Face (private)

In [ ]:
from google.colab import userdata
from datasets import Dataset, DatasetDict, Features, Value, ClassLabel
from huggingface_hub import DatasetCard, HfApi

token = userdata.get("HF_TOKEN")
api = HfApi(token=token)

def tag(name):
    try:
        api.create_tag(REPO_ID, tag=name, repo_type="dataset")
        print(f"tagged {name}")
    except Exception as e:
        print(f"tag {name} not created ({type(e).__name__}): it probably exists already")

# Keep the version people may already use downloadable BEFORE overwriting it
if api.repo_exists(REPO_ID, repo_type="dataset"):
    tag(PREVIOUS)

features = Features({"text": Value("string"), "label": ClassLabel(names=["benign", "injection"]),
                     "source": Value("string"), "lang": Value("string"), "family": Value("string"),
                     "origin": Value("string"), "license": Value("string")})
dd = DatasetDict({name: Dataset.from_pandas(g.drop(columns="split").reset_index(drop=True),
                                            features=features, preserve_index=False)
                  for name, g in df.groupby("split")})
print(dd)
dd.push_to_hub(REPO_ID, private=PRIVATE, token=token, commit_message=f"Dataset {VERSION}")
DatasetCard(card_md).push_to_hub(REPO_ID, repo_type="dataset", token=token, commit_message=f"Card {VERSION}")
tag(VERSION)
print(f"\nPublished: https://huggingface.co/datasets/{REPO_ID}")

## 5. Check it loads back

In [ ]:
from datasets import load_dataset
check = load_dataset(REPO_ID, split="validation", revision=VERSION, token=token)
print(check)
print("harmless emails in validation:", sum(1 for r in check if r["label"] == 0 and "SUBJECT:" in r["text"]))
old = load_dataset(REPO_ID, split="validation", revision=PREVIOUS, token=token)
print(f"{PREVIOUS} is still available: {len(old):,} validation rows")

## 6. Check the page

1. Open the link printed in section 4. The card should say **v0.2.1** and have a **Changelog** section.
2. Under *Files and versions*, the branch menu should list the tags `v0.2` and `v0.2.1`.